# 02 -- Weight vs Activation Decomposition

Loads the weight-only / activation-only / full quantization accuracy decomposition and
reproduces Table 2 of the report (Sec. 5.2), plus a grouped bar chart of weight_loss_pts
vs activation_loss_pts per combination (optional figure referenced in the report text).

Source CSV: `results/20260816_230437_38678/csv/activation_decomposition.csv`
(CIFAR10 only -- no ImageNet100 rows exist in this file; see report Sec. 5.2/6.3 for the
limitation this implies).


In [1]:
# Requirements: pandas==3.0.5, numpy==2.5.1, matplotlib==3.11.1, seaborn==0.13.2, scipy==1.18.0
# All notebooks in this report use the same environment; paths below are relative to
# report/notebooks/, so the notebook must be run with its own directory as the working
# directory (the default for `jupyter nbconvert --execute` and for Jupyter's own kernel).
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO = "../.."  # report/notebooks -> report -> repo root
FIG_DIR = "../figures"
import os
os.makedirs(FIG_DIR, exist_ok=True)


In [2]:
RUN = f"{REPO}/results/20260816_230437_38678/csv"
decomp = pd.read_csv(f"{RUN}/activation_decomposition.csv")
decomp


,model,dataset,stage,fp32_acc,weights_only_acc,activations_only_acc,full_acc,weight_loss_pts,activation_loss_pts,total_loss_pts,dominant_source
0,cnn,CIFAR10,PTQ,81.30,71.06,81.37,71.05,10.24,-0.07,10.25,weight
1,cnn,CIFAR10,QAT,81.30,84.52,78.19,84.45,-3.22,3.11,-3.15,activation
2,resnet18_no_weights,CIFAR10,PTQ,84.18,82.77,84.09,82.78,1.41,0.09,1.40,weight
3,resnet18_no_weights,CIFAR10,QAT,84.18,90.54,89.66,90.58,-6.36,-5.48,-6.40,activation
4,resnet50_no_weights,CIFAR10,PTQ,81.14,79.24,81.37,79.53,1.90,-0.23,1.61,weight
5,resnet50_no_weights,CIFAR10,QAT,81.14,86.64,84.86,86.66,-5.50,-3.72,-5.52,activation


Format model/stage labels and select the columns that make up report Table 2.


In [3]:
MODEL_LABEL = {"cnn": "CNN", "resnet18_no_weights": "ResNet-18", "resnet50_no_weights": "ResNet-50"}
decomp["Modell"] = decomp["model"].map(MODEL_LABEL)
decomp["dominant_de"] = decomp["dominant_source"].map({"weight": "Gewicht", "activation": "Aktivierung"})

table2 = decomp[["Modell", "stage", "weight_loss_pts", "activation_loss_pts", "dominant_de"]].copy()
table2.columns = ["Modell", "Stufe", "Gewicht", "Aktivierung", "Dominant"]
for c in ["Gewicht", "Aktivierung"]:
    table2[c] = table2[c].round(2)
table2.to_csv(f"{FIG_DIR}/tab_02_weight_activation_decomposition.csv", index=False)
table2


,Modell,Stufe,Gewicht,Aktivierung,Dominant
0,CNN,PTQ,10.24,-0.07,Gewicht
1,CNN,QAT,-3.22,3.11,Aktivierung
2,ResNet-18,PTQ,1.41,0.09,Gewicht
3,ResNet-18,QAT,-6.36,-5.48,Aktivierung
4,ResNet-50,PTQ,1.90,-0.23,Gewicht
5,ResNet-50,QAT,-5.50,-3.72,Aktivierung


Optional bar chart: weight_loss_pts vs activation_loss_pts per model, split by PTQ/QAT, to visualize the sign flip between stages discussed in Sec. 6.3 of the report.


In [4]:
# Two-panel bar chart (PTQ left, QAT right); one bar group per model, two bars
# (weight/activation loss) per group -- shows the dominant-source flip between stages.
fig, axes = plt.subplots(1, 2, figsize=(6.5, 3.0), sharey=True, layout="constrained")
colors = {"Gewicht": "#4C72B0", "Aktivierung": "#DD8452"}
for ax, stage in zip(axes, ["PTQ", "QAT"]):
    sub = table2[table2["Stufe"] == stage]
    x = np.arange(len(sub))
    width = 0.35
    ax.bar(x - width / 2, sub["Gewicht"], width, label="Gewicht", color=colors["Gewicht"])
    ax.bar(x + width / 2, sub["Aktivierung"], width, label="Aktivierung", color=colors["Aktivierung"])
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(sub["Modell"], fontsize=8)
    ax.set_title(stage, fontsize=9)
    ax.set_ylabel("Verlust (Prozentpunkte)" if stage == "PTQ" else "")
axes[0].legend(fontsize=7, frameon=False)
fig.savefig(f"{FIG_DIR}/fig_opt_weight_activation_bars.pdf")
fig.savefig(f"{FIG_DIR}/fig_opt_weight_activation_bars.png", dpi=200)
plt.close(fig)


## Output

- `figures/tab_02_weight_activation_decomposition.csv` -- report Table 2
- `figures/fig_opt_weight_activation_bars.pdf` / `.png` -- optional bar chart (Sec. 6.3)
